# LIMPIEZA DE DATOS

## PASO 1:  Importacion y carga Inicial

In [2]:
import pandas as pd


df = pd.read_csv('dataset proceso residentado medico - CONSOLIDADO.csv')
print(f"El dataset contiene {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head(3))

El dataset contiene 22329 filas y 26 columnas.


,Año,No,CMP,No Doc.,Apellido Paterno,Apellido Materno,Nombres,Universidad,Tipo,Especialidad/SubEspecialidad,...,ENAM,Examen,Not. Final,Obs.,Ingresó si/no,Universidad (INGRESO),Tipo (INGRESO),Especialidad/SubEspecialidad (INGRESO),Modalidad (INGRESO),Sede (INGRESO)
0,2023,1,93552,73815684,YAURI,HUAMAN,MARTIN EFREN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.5 pts.,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,88328,71701411,CARBAJAL,DIAZ,MARIANA DEL CARMEN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,2 pts.,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,95865,46024669,BUSTAMANTE,CARDENAS,EDGAR MIGUEL,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1 pts.,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins


## Paso 2 : Exploracion y estandarizacion de columnas

#### Antes de limpiar los valores internos, es una buena práctica estandarizar los nombres de las columnas para evitar errores tipográficos al codificar.

In [3]:
df.info()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '')
print("\nNuevas columnas:", df.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 22329 entries, 0 to 22328
Data columns (total 26 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Año                                     22329 non-null  int64  
 1   No                                      22329 non-null  int64  
 2   CMP                                     22329 non-null  str    
 3   No Doc.                                 22329 non-null  int64  
 4     Apellido Paterno                      22328 non-null  str    
 5   Apellido Materno                        22323 non-null  str    
 6   Nombres                                 22329 non-null  str    
 7   Universidad                             22329 non-null  str    
 8   Tipo                                    22329 non-null  str    
 9   Especialidad/SubEspecialidad            22329 non-null  str    
 10  Modalidad                               22329 non-null  str    
 11  

## Paso 3: Tratamiento de duplicados

### Para asegurar la exactitud de los datos, eliminaremos cualquier registro que se haya duplicado en la consolidación del archivo.

In [4]:
duplicados_previos = df.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados_previos}")

df = df.drop_duplicates()

Filas duplicadas encontradas: 0


## Paso 4 : Limpieza profunda y transformacion de tipos de datos

### Se extraeran solo los numeros y no las letras y se colocara cero a las columnas vacias de bonificacion porque no obtubieron puntos extra

In [5]:
# 1. Limpieza de las columnas de bonificación (esta parte se  rellena con 0)
columnas_bonificacion = ['sncds','serum', '1er_niv', '5to_sup']

def limpiar_puntajes_bonos(valor):
    if pd.isna(valor):
        return 0.0 # Rellenamos con 0 a quienes no tienen bono
    valor_str = str(valor).replace('pts.', '').replace('pts', '').replace(',', '').strip()
    try:
        return float(valor_str)
    except ValueError:
        return 0.0

for col in columnas_bonificacion:
    if col in df.columns: 
        df[col] = df[col].apply(limpiar_puntajes_bonos)


# 2. Limpieza de notas principales quitando el texto, pero MANTENIENDO los nulos (NaN)
columnas_notas = ['prom_pre', 'enam', 'examen']

for col in columnas_notas:
    if col in df.columns:
        # Forzamos a que sea texto, reemplazamos 'pts.' y limpiamos espacios vacíos
        df[col] = df[col].astype(str).str.replace('pts.', '', regex=False)\
                                     .str.replace('pts', '', regex=False)\
                                     .str.strip()
        
        # pd.to_numeric con errors='coerce' convierte el texto limpio a decimal (float). 
        # Si encuentra un vacío ("nan"), lo deja matemáticamente nulo (NaN) en lugar de poner cero.
        df[col] = pd.to_numeric(df[col], errors='coerce')


df.head(3)

,año,no,cmp,no_doc,apellido_paterno,apellido_materno,nombres,universidad,tipo,especialidad/subespecialidad,...,enam,examen,not_final,obs,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso)
0,2023,1,93552,73815684,YAURI,HUAMAN,MARTIN EFREN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.5,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,88328,71701411,CARBAJAL,DIAZ,MARIANA DEL CARMEN,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,2.0,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,95865,46024669,BUSTAMANTE,CARDENAS,EDGAR MIGUEL,CONAREME,Esp.,ANATOMIA PATOLOGICA,...,1.0,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins


## Paso 5 : Eliminar datos privados

### Aqui eliminaremos las columnas de los datos personales 

In [6]:
columnas_personales = ['apellido_paterno', 'apellido_materno', 'nombres','cmp']
df.drop(columns=columnas_personales, inplace=True, errors='ignore')
print("Columnas actuales en el dataset:")
df.head(5)

Columnas actuales en el dataset:


,año,no,no_doc,universidad,tipo,especialidad/subespecialidad,modalidad,serum,sncds,1er_niv,...,enam,examen,not_final,obs,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso)
0,2023,1,73815684,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,...,1.5,60.4,73.870,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,2,71701411,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,...,2.0,57.2,71.173,NaN,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,3,46024669,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,...,1.0,56.0,69.893,NaN,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
3,2023,4,46374473,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,...,1.5,56.0,69.346,NaN,SI,UNFV,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
4,2023,5,76428325,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,...,1.0,56.0,68.849,NaN,SI,UPCH,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Cayetano Heredia


## Paso 6: Eliminar columnas no necesarias

### Eliminamos las columnas numero de orden y observaciones

In [7]:
columnas_a_eliminar = ["no", "obs"]
df.drop(columns=columnas_a_eliminar, inplace=True, errors="ignore")

print("Se eliminaron las columnas 'no' y 'obs' (si existían).")
df.head(5)

Se eliminaron las columnas 'no' y 'obs' (si existían).


,año,no_doc,universidad,tipo,especialidad/subespecialidad,modalidad,serum,sncds,1er_niv,5to_sup,prom_pre,enam,examen,not_final,ingresó_si/no,universidad_(ingreso),tipo_(ingreso),especialidad/subespecialidad_(ingreso),modalidad_(ingreso),sede_(ingreso)
0,2023,73815684,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.970,1.5,60.4,73.870,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
1,2023,71701411,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.973,2.0,57.2,71.173,SI,UNMSM,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
2,2023,46024669,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,1.0,1.893,1.0,56.0,69.893,SI,URP,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
3,2023,46374473,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.846,1.5,56.0,69.346,SI,UNFV,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Edgardo Rebagliati Martins
4,2023,76428325,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.849,1.0,56.0,68.849,SI,UPCH,Esp.,ANATOMIA PATOLOGICA,Libre,Hospital Nacional Cayetano Heredia


## Paso 7 : Crear variable de ponderacion

### Representaría la ponderación del currículum vitae u otros factores evaluativos aparte del examen

In [9]:
# 1. Asegurarnos de que 'not_final' esté limpia y sea numérica (por si tiene el texto "pts")
df['not_final'] = df['not_final'].astype(str).str.replace('pts.', '', regex=False)\
                                 .str.replace('pts', '', regex=False)\
                                 .str.strip()

# Convertir a valor numérico decimal (float)
df['not_final'] = pd.to_numeric(df['not_final'], errors='coerce')

# 2. Crear la nueva columna 'nota_cv' (Nota Final - Examen)
df['nota_cv'] = df['not_final'] - df['examen']

# 3. Mostrar las primeras filas de estas 3 columnas para verificar que la resta se hizo bien
print("Verificación de la nueva variable 'nota_cv':")
display(df[['examen', 'not_final', 'nota_cv']].head())

Verificación de la nueva variable 'nota_cv':


,examen,not_final,nota_cv
0,60.4,73.870,13.470
1,57.2,71.173,13.973
2,56.0,69.893,13.893
3,56.0,69.346,13.346
4,56.0,68.849,12.849


## Paso 8: Creacion de filtro para comparar columnas

### Compararemos las columnas originales con las columnas de ingreso

In [11]:
# 1. Crear un filtro principal solo con los postulantes que lograron ingresar
df_ingresantes = df[df['ingresó_si/no'] == 'SI']

# 2. Definir los pares de columnas a comparar (Original vs Ingreso)
pares_comparacion = [
    ('universidad', 'universidad_(ingreso)'),
    ('tipo', 'tipo_(ingreso)'),
    ('especialidad/subespecialidad', 'especialidad/subespecialidad_(ingreso)'),
    ('modalidad', 'modalidad_(ingreso)')
]

columnas_a_eliminar = []

print("--- Análisis de Diferencias en Postulantes Ingresantes ---")

# 3. Iterar sobre cada par para compararlos
for col_original, col_ingreso in pares_comparacion:
    # Verificamos que ambas columnas existan en el dataset
    if col_original in df.columns and col_ingreso in df.columns:
        
        # Creamos una copia temporal de los ingresantes para aplicar filtros específicos
        df_comparar = df_ingresantes.copy()
        
        print(f"\nComparando: '{col_original}' vs '{col_ingreso}'")
        
        # --- FILTRO EXTRA PARA UNIVERSIDAD ---
        if col_original == 'universidad':
            # Excluimos los registros donde la universidad de origen sea CONAREME
            filtro_conareme = df_comparar['universidad'].astype(str).str.strip().str.upper() != 'CONAREME'
            df_comparar = df_comparar[filtro_conareme]
            print(f"[*] Filtro aplicado: Se excluyeron postulantes con origen 'CONAREME' para esta evaluación.")
        
        # Calculamos el total de registros válidos para esta comparación en particular
        total_evaluados = len(df_comparar)
        
        if total_evaluados == 0:
            print("No hay registros válidos para comparar después de aplicar los filtros.")
            continue
            
        # Estandarizamos a mayúsculas y sin espacios para una comparación justa
        val_original = df_comparar[col_original].astype(str).str.strip().str.upper()
        val_ingreso = df_comparar[col_ingreso].astype(str).str.strip().str.upper()
        
        # Contamos cuántas filas son diferentes
        diferencias = (val_original != val_ingreso).sum()
        porcentaje_dif = (diferencias / total_evaluados) * 100
        
        print(f"Registros evaluados: {total_evaluados}")
        print(f"Registros diferentes: {diferencias} ({porcentaje_dif:.2f}%)")
        
        # 4. Lógica de eliminación: Si son 100% idénticas, la marcamos para borrar
        if diferencias == 0:
            print(f"-> ¡Son idénticas! Es seguro eliminar la columna '{col_ingreso}'.")
            columnas_a_eliminar.append(col_ingreso)
        else:
            print(f"-> Hay diferencias. NO se recomienda eliminar '{col_ingreso}'.")

--- Análisis de Diferencias en Postulantes Ingresantes ---

Comparando: 'universidad' vs 'universidad_(ingreso)'
[*] Filtro aplicado: Se excluyeron postulantes con origen 'CONAREME' para esta evaluación.
Registros evaluados: 6573
Registros diferentes: 815 (12.40%)
-> Hay diferencias. NO se recomienda eliminar 'universidad_(ingreso)'.

Comparando: 'tipo' vs 'tipo_(ingreso)'
Registros evaluados: 8817
Registros diferentes: 0 (0.00%)
-> ¡Son idénticas! Es seguro eliminar la columna 'tipo_(ingreso)'.

Comparando: 'especialidad/subespecialidad' vs 'especialidad/subespecialidad_(ingreso)'
Registros evaluados: 8817
Registros diferentes: 1282 (14.54%)
-> Hay diferencias. NO se recomienda eliminar 'especialidad/subespecialidad_(ingreso)'.

Comparando: 'modalidad' vs 'modalidad_(ingreso)'
Registros evaluados: 8817
Registros diferentes: 0 (0.00%)
-> ¡Son idénticas! Es seguro eliminar la columna 'modalidad_(ingreso)'.


## Paso 9: Procedemos a elimanr columnas

### eliminaremos la columna modalidad(ingreso) y tipo(ingreso) del datset


In [12]:
# Eliminar las columnas redundantes del DataFrame principal
if columnas_a_eliminar:
    df.drop(columns=columnas_a_eliminar, inplace=True)
    print(f"\nResumen: Se eliminaron con éxito las columnas redundantes: {columnas_a_eliminar}")
else:
    print("\nResumen: No se eliminó ninguna columna porque todas presentan diferencias útiles.")

# Visualizar cómo quedó el dataset después de esta limpieza avanzada
print(f"El dataset contiene {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head(3))


Resumen: Se eliminaron con éxito las columnas redundantes: ['tipo_(ingreso)', 'modalidad_(ingreso)']
El dataset contiene 22329 filas y 20 columnas.


,año,no_doc,universidad,tipo,especialidad/subespecialidad,modalidad,serum,sncds,1er_niv,5to_sup,prom_pre,enam,examen,not_final,ingresó_si/no,universidad_(ingreso),especialidad/subespecialidad_(ingreso),sede_(ingreso),total_bonificacion,nota_cv
0,2023,73815684,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.970,1.5,60.4,73.870,SI,UNMSM,ANATOMIA PATOLOGICA,Hospital Nacional Edgardo Rebagliati Martins,10.0,13.470
1,2023,71701411,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,0.0,1.973,2.0,57.2,71.173,SI,UNMSM,ANATOMIA PATOLOGICA,Hospital Nacional Edgardo Rebagliati Martins,10.0,13.973
2,2023,46024669,CONAREME,Esp.,ANATOMIA PATOLOGICA,Libre,10.0,0.0,0.0,1.0,1.893,1.0,56.0,69.893,SI,URP,ANATOMIA PATOLOGICA,Hospital Nacional Edgardo Rebagliati Martins,11.0,13.893


## Paso 10: Reemplazar las abreviaturas de la columna tipo

### se remplazara por "Especialidad" y "Subespecialidad"

In [13]:
# Reemplazar las abreviaturas en la columna 'tipo' por su nombre completo
df['tipo'] = df['tipo'].replace({
    'Esp.': 'Especialidad',
    'Sub.': 'Subespecialidad'
})

# Verificar el cambio
print(df['tipo'].value_counts())

tipo
Especialidad       21243
Subespecialidad     1086
Name: count, dtype: int64


## Paso 11: Exportar a dataset limpio

### En este paso guardamos el DataFrame ya depurado en un archivo nuevo para conservar los datos limpios y no sobrescribir el archivo original

In [14]:
df.to_csv('dataset_limpio2.csv', index=False, encoding='utf-8-sig')
print('Nuevo dataset creado: dataset_limpio2.csv')

Nuevo dataset creado: dataset_limpio2.csv
